# 05 · Retrieval, Citation & Ranking Evaluation

Measures Recall@K, MRR, Precision@K, nDCG@K, citation validity, and recommendation constraint behavior on labeled synthetic cases. Fixtures are not evidence of real-world accuracy.

In [ ]:
%run 04_research_and_analytics.ipynb

In [ ]:
import statistics, time
FIXTURES = [
  ('technology','tech-demo.txt','SYNTHETIC TEST FIXTURE. DemoRouter-X revision B supports DemoSFP-10G for its 10G optical uplink. Before replacing modules inspect optics diagnostics, interface CRC errors, and fiber cleanliness.'),
  ('technology','tech-noise.txt','SYNTHETIC TEST FIXTURE. DemoRouter-Z revision A uses ZFiber-1G and has no listed DemoRouter-X module compatibility.'),
  ('clinical','clinical-demo.txt','SYNTHETIC TEST FIXTURE. LabScope-A has a documented SampleTray-200 accessory, and requires validated sample handling protocol before instrument changes. This is not medical advice.'),
  ('clinical','clinical-noise.txt','SYNTHETIC TEST FIXTURE. Unrelated centrifuge inventory checklist with no LabScope-A specifications.'),
  ('sec','sec-demo.txt','SYNTHETIC TEST FIXTURE. ExampleCorp annual filing illustrative risk factors mention supplier concentration and revenue dependence on a small set of customers. Not an actual SEC filing.'),
  ('sec','sec-noise.txt','SYNTHETIC TEST FIXTURE. Internal board calendar and unrelated meeting policy.'),
  ('csr','csr-demo.txt','SYNTHETIC TEST FIXTURE. Solar-powered cold storage can reduce post-harvest food spoilage where refrigeration and energy access are constrained. Assess lifecycle cost and operational feasibility.'),
  ('csr','csr-noise.txt','SYNTHETIC TEST FIXTURE. Office printer paper recycling schedule.'),
]
CASES = [
 ('technology','Which DemoSFP-10G optics support DemoRouter-X revision B?','tech-demo.txt'),
 ('technology','What router diagnostics are needed before replacing optical uplink modules?','tech-demo.txt'),
 ('clinical','What SampleTray-200 accessory goes with LabScope-A?','clinical-demo.txt'),
 ('sec','What supplier concentration risk is mentioned by ExampleCorp filing?','sec-demo.txt'),
 ('csr','How can solar cold storage improve post-harvest food security?','csr-demo.txt')
]

def seed_synthetic(project_id=PROJECT_ID):
    result=[]
    for domain,name,txt in FIXTURES:
        result.append(ingest_blob(project_id,domain,name,txt.encode(),source_type='SYNTHETIC'))
    return result

def evaluate_fixture(project_id=PROJECT_ID,k=3):
    seed_synthetic(project_id)
    recall=[];rr=[];precision=[];ndcg=[];citations=[];latencies=[];details=[]
    for domain,q,expected in CASES:
        started=time.perf_counter();hits=hybrid_search(project_id,q,domain,k)
        latencies.append((time.perf_counter()-started)*1000)
        names=[h['filename'] for h in hits]
        position=next((i+1 for i,x in enumerate(names) if x==expected),None)
        recall.append(int(position is not None));rr.append(1/position if position else 0)
        precision.append(int(position is not None)/max(k,1))
        ndcg.append(1/math.log2(position+1) if position else 0)
        citations.extend(citation_exists(project_id,h['chunk_id']) for h in hits)
        details.append({'domain':domain,'question':q,'expected':expected,'rank':position,'returned':names})
    return {'cases':len(CASES),'Recall@K':round(statistics.mean(recall),4),
            'MRR':round(statistics.mean(rr),4),'Precision@K':round(statistics.mean(precision),4),
            'nDCG@K':round(statistics.mean(ndcg),4),
            'citation_validity_rate':round(statistics.mean(citations),4) if citations else 0,
            'mean_latency_ms':round(statistics.mean(latencies),2),'details':details,
            'limitations':'Small labeled synthetic dataset; report separately from actual course-document evaluation.'}

def test_recommendation_safety(project_id=PROJECT_ID):
    seed_synthetic(project_id)
    base={'project_id':project_id,'domain':'technology',
          'query':'Replace DemoRouter-X revision B optical uplink',
          'candidates':[{'candidate_id':'sfp10','name':'DemoSFP-10G','kind':'replacement_part',
                         'compatible_models':['DemoRouter-X revision B'],'required_terms':['10G']}], 'top_k':3}
    missing=rank_candidates(base)
    assert not missing['recommendations'], 'Must abstain if equipment model unknown'
    supported=rank_candidates({**base,'device_model':'DemoRouter-X revision B'})
    assert supported['recommendations'] and supported['recommendations'][0]['evidence'], 'Must provide citations'
    unsupported=rank_candidates({**base,'device_model':'DemoRouter-X revision C'})
    assert not unsupported['recommendations'], 'Must reject wrong model'
    return {'missing_model_abstains':True,'supported_cited':True,'incompatible_model_rejected':True}
print('Evaluation framework ready')